
# Download AutoLaparo to Google Drive

Downloads the **AutoLaparo** dataset
([Wang et al., MICCAI 2022](https://arxiv.org/abs/2208.05314),
[autolaparo.github.io](https://autolaparo.github.io/)) -- 21 laparoscopic
hysterectomy videos with phase/motion/segmentation labels -- from a personal
access link obtained via AutoLaparo's request form:

```
http://210.3.251.30:8080/share.cgi?ssid=07LR4TG
```

**This link is unverified.** It looks like a QNAP NAS share link
(`share.cgi?ssid=...`), a pattern that can mean either "this URL *is* the
direct file download" or "this URL is an HTML page listing one or more
files to click through" -- direct testing wasn't possible from this
notebook's build environment (the connection timed out reaching that host/port
from here). So this notebook is split into two stages: **Stage 1 fetches the
page and shows you what's actually there** before Stage 2 tries to download
anything. Read Stage 1's output before trusting Stage 2 on the first run.

**Treat this URL like a credential**: it's a personal access link tied to
your dataset request, not a public bucket -- don't commit it or post it
publicly.

## What this notebook does

1. Mounts Drive (Colab) and sets up `data/autolaparo/`.
2. **Stage 1**: fetches the share URL, prints what came back (a file? an
   HTML listing?), and parses out candidate file links if it's a listing
   page.
3. **Stage 2**: downloads each discovered file with resume support and a
   post-download sanity check (rejects anything that's actually an HTML
   error page saved with the wrong extension -- a real risk with NAS shares
   that return a login/error page instead of the file).
4. Extracts any archive files found.


## 0. Setup: paths and packages

In [ ]:

import sys, subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/endofm-benchmark")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests", "tqdm", "beautifulsoup4"], check=True)
else:
    PROJECT_ROOT = Path(r"E:/endofm-benchmark")

DATA_DIR = PROJECT_ROOT / "data" / "autolaparo"
RAW_DIR = DATA_DIR / "raw"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)
print("Data dir:", DATA_DIR)


## 1. Stage 1 -- fetch the share page and see what it actually is

Sends a browser-like `User-Agent` (some NAS setups block default
`requests`/`curl` user agents). Then branches on the response:

- If the response looks like **HTML**, it's a listing/landing page --
  the raw HTML is saved to `share_page.html` (open it if the parser below
  finds nothing useful) and parsed for candidate file links (common QNAP
  patterns: hrefs containing `ssid=`, `fid=`, `download`, or ending in a
  known archive/video extension).
- If it looks like **binary data already**, the share URL itself is the
  direct file -- no listing page exists.

**Read the printed output before running Stage 2.**

In [ ]:

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

SHARE_URL = "http://210.3.251.30:8080/share.cgi?ssid=07LR4TG"

BROWSER_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
}

FILE_EXTS = (".zip", ".rar", ".7z", ".tar", ".tar.gz", ".mp4", ".avi", ".mov", ".mkv", ".csv", ".json")

session = requests.Session()
session.headers.update(BROWSER_HEADERS)

resp = session.get(SHARE_URL, timeout=60, allow_redirects=True, stream=True)
content_type = resp.headers.get("Content-Type", "")
content_length = resp.headers.get("Content-Length", "unknown")
print(f"status={resp.status_code}  content-type={content_type!r}  content-length={content_length}")
print(f"final URL after redirects: {resp.url}")

candidate_links = []

if "text/html" in content_type.lower():
    html = resp.text
    (DATA_DIR / "share_page.html").write_text(html, encoding="utf-8")
    print(f"\nThis is an HTML page ({len(html)} chars) -- saved to {DATA_DIR / 'share_page.html'}")

    if resp.status_code != 200 or "password" in html.lower():
        print("WARNING: response may be an error page or a password prompt -- "
              "open share_page.html and check manually before proceeding.")

    soup = BeautifulSoup(html, "html.parser")
    seen = set()
    for a in soup.find_all("a", href=True):
        href = a["href"]
        text = a.get_text(strip=True)
        is_candidate = (
            href.lower().endswith(FILE_EXTS)
            or "ssid=" in href or "fid=" in href or "download" in href.lower()
        )
        if is_candidate and href not in seen:
            seen.add(href)
            candidate_links.append((text or "(no link text)", urljoin(resp.url, href)))

    print(f"\nFound {len(candidate_links)} candidate file link(s):")
    for text, url in candidate_links:
        print(f"  - {text!r:40s} -> {url}")

    if not candidate_links:
        print("\nNo candidates auto-detected. Open share_page.html in a browser "
              "(or read its source below) and manually set `candidate_links` "
              "in the next cell before running Stage 2.")
        print("\n--- first 3000 chars of the page ---")
        print(html[:3000])
else:
    print("\nResponse does not look like HTML -- treating SHARE_URL itself as a direct file download.")
    candidate_links = [("autolaparo (direct)", SHARE_URL)]

resp.close()


## 2. Manual override (only if Stage 1 found nothing useful)

If Stage 1 printed 0 candidates, inspect `share_page.html` and fill this in
by hand, e.g.:

```python
candidate_links = [
    ("autolaparo_videos.zip", "http://210.3.251.30:8080/download.cgi?ssid=07LR4TG&fid=..."),
]
```

If Stage 1 already found the right link(s), just leave this cell as-is --
it won't override anything.

In [ ]:

# candidate_links = [...]   # uncomment and fill in only if Stage 1 found nothing
print(f"Proceeding with {len(candidate_links)} link(s).")


## 3. Stage 2 -- resumable download + sanity check

Same resume-on-`.part` pattern as the Cholec80 notebook. The extra check
here: after downloading, verifies the file **isn't actually an HTML error
page saved under the wrong name** (a real risk with NAS shares -- a login
wall or expired-link error page would otherwise look like a "successful"
download). A `.verified` marker (keyed to file size) skips re-checking a
file that hasn't changed.

In [ ]:

from tqdm.auto import tqdm

def guess_filename(url, link_text):
    from urllib.parse import urlparse, parse_qs
    parsed = urlparse(url)
    qs = parse_qs(parsed.query)
    for key in ("fname", "filename", "name"):
        if key in qs:
            return qs[key][0]
    if link_text and any(link_text.lower().endswith(ext) for ext in FILE_EXTS):
        return link_text
    return f"autolaparo_{qs.get('fid', qs.get('ssid', ['file']))[0]}.bin"

def looks_like_html(path, sniff_bytes=512):
    with open(path, "rb") as f:
        head = f.read(sniff_bytes).lstrip().lower()
    return head.startswith(b"<!doctype html") or head.startswith(b"<html") or b"<head" in head[:200]

def download_resumable(url, dest, part_suffix=".part", chunk_mb=8):
    part_path = dest.with_suffix(dest.suffix + part_suffix)
    resume_pos = part_path.stat().st_size if part_path.exists() else 0
    headers = {"Range": f"bytes={resume_pos}-"} if resume_pos else {}

    with session.get(url, headers=headers, stream=True, timeout=60) as r:
        if resume_pos and r.status_code == 200:
            print("  server doesn't support resuming; restarting from 0.")
            resume_pos = 0
            part_path.unlink()
        r.raise_for_status()
        total = resume_pos + int(r.headers.get("content-length", 0))
        mode = "ab" if resume_pos else "wb"
        with open(part_path, mode) as f, tqdm(
            total=total or None, initial=resume_pos, unit="B", unit_scale=True,
            unit_divisor=1024, desc=dest.name,
        ) as pbar:
            for chunk in r.iter_content(chunk_size=chunk_mb * 1024 * 1024):
                f.write(chunk)
                pbar.update(len(chunk))

    part_path.rename(dest)

downloaded_paths = []
for link_text, url in candidate_links:
    fname = guess_filename(url, link_text)
    dest = RAW_DIR / fname
    marker = RAW_DIR / f"{fname}.verified"

    if dest.exists() and marker.exists() and marker.read_text().strip() == str(dest.stat().st_size):
        print(f"Already downloaded + verified: {dest}")
        downloaded_paths.append(dest)
        continue

    print(f"Downloading: {url} -> {dest}")
    download_resumable(url, dest)

    if looks_like_html(dest):
        print(f"  REJECTED: {dest.name} looks like an HTML page, not real data "
              f"(expired link, login wall, or error page). Not marking as verified.")
        print(f"  First 500 bytes saved for inspection: {dest}")
        continue

    marker.write_text(str(dest.stat().st_size))
    print(f"  OK: {dest} ({dest.stat().st_size / 1e9:.3f} GB)")
    downloaded_paths.append(dest)

print(f"\n{len(downloaded_paths)} file(s) downloaded and passed the sanity check.")


## 4. Extract any archives

In [ ]:

import zipfile, tarfile

ARCHIVE_EXTS = {".zip": zipfile.ZipFile, ".tar": tarfile.open, ".gz": tarfile.open}

for path in downloaded_paths:
    suffix = "".join(path.suffixes[-2:]) if path.suffix == ".gz" else path.suffix
    extract_dir = RAW_DIR / path.stem
    if path.suffix.lower() == ".zip":
        if extract_dir.exists() and any(extract_dir.rglob("*")):
            print("Already extracted:", extract_dir)
            continue
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(path) as zf:
            bad = zf.testzip()
            if bad is not None:
                print(f"CORRUPT zip, skipping extraction: {path} (bad member: {bad})")
                continue
            for member in tqdm(zf.infolist(), desc=f"Extracting {path.name}"):
                zf.extract(member, extract_dir)
        print("Extracted:", extract_dir)
    elif path.suffix.lower() in (".tar", ".gz"):
        if extract_dir.exists() and any(extract_dir.rglob("*")):
            print("Already extracted:", extract_dir)
            continue
        extract_dir.mkdir(parents=True, exist_ok=True)
        with tarfile.open(path) as tf:
            tf.extractall(extract_dir)
        print("Extracted:", extract_dir)
    else:
        print("Not an archive, left as-is:", path)


## Notes / caveats

- **Unverified page structure**: this notebook's Stage 1/Stage 2 split and
  the QNAP-style link-detection patterns are best-effort, built without
  being able to load the actual share page from the build environment. If
  Stage 1 finds 0 candidates and the response isn't a direct file either,
  open `share_page.html` and adjust the parsing logic or the manual
  override in Section 2.
- **Not in Endo-FM's pretraining data**: unlike Kvasir-Capsule, AutoLaparo
  is not listed in Endo-FM's own pretraining-data table -- a benchmark on
  it is a cleaner generalization test than the Kvasir-Capsule one, per
  earlier discussion.
- This notebook only fetches raw data -- building the phase-recognition
  linear-probe pipeline on top of it (frame extraction, label loading,
  feature extraction) is a follow-up step, analogous to
  `benchmark_endofm_kvasir_capsule.ipynb`.
